# Перевірка знань: Pydantic

**20 питань.** Notebook складається з трьох частин:

- **Частина 1 (1–8):** теоретичні питання — напишіть відповідь у клітинці нижче питання.
- **Частина 2 (9–15):** знайдіть помилку — поясніть, у чому проблема, і напишіть виправлений код у порожній клітинці.
- **Частина 3 (16–20):** що виведе код — спочатку напишіть свою відповідь, **не запускаючи** клітинку, потім перевірте себе.

Під кожним питанням є згорнутий блок **«💡 Відповідь»** — відкривайте його лише після того, як дали власну відповідь!

> Усе на **Pydantic v2**. Перед роботою виконайте клітинку нижче.

Успіхів! 🍀

In [1]:
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Optional, Literal
import pydantic

print("Pydantic версія:", pydantic.VERSION)

Pydantic версія: 2.13.4


---
## Частина 1. Теорія

### Питання 1
Що таке **Pydantic** і чим клас-нащадок `BaseModel` відрізняється від звичайного Python-класу?

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<b>Pydantic</b> — бібліотека для <b>валідації даних</b> на основі анотацій типів. Ми описуємо клас, успадкований від <code>BaseModel</code>, лише перелічуючи поля з типами.

Відмінності від звичайного класу:
<ul>
<li><code>__init__</code> генерується <b>автоматично</b> (як у <code>@dataclass</code>);</li>
<li>дані <b>перевіряються й приводяться</b> до типів при створенні об'єкта;</li>
<li>безкоштовно отримуємо зручний <code>__repr__</code>, серіалізацію в dict/JSON, JSON-схему.</li>
</ul>

Звичайний клас прийме будь-що (вік рядком — без проблем), а помилка спливе пізніше й у дивному місці.

</details>

### Питання 2
Що таке **валідація** в Pydantic? У який момент вона відбувається і що станеться з некоректними даними?

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<b>Валідація</b> — перевірка, що дані відповідають оголошеним типам та обмеженням, із <b>розумним приведенням</b> (наприклад, рядок <code>"20"</code> → число <code>20</code>).

Вона відбувається <b>при створенні об'єкта</b> (і при <code>model_validate</code>). Якщо дані некоректні й привести їх неможливо — Pydantic піднімає <code>ValidationError</code> одразу, <b>на «кордоні»</b> програми, а не глибоко в логіці.

Pydantic збирає <b>всі</b> помилки разом, а не падає на першій.

</details>

### Питання 3
Що робить функція `Field()`? Наведіть приклади 2–3 обмежень, які можна через неї задати.

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<code>Field()</code> дозволяє налаштувати поле: задати значення за замовчуванням, обмеження та опис.

Приклади обмежень:
<ul>
<li><code>gt</code>, <code>ge</code>, <code>lt</code>, <code>le</code> — більше/більше-рівне/менше/менше-рівне за число;</li>
<li><code>min_length</code>, <code>max_length</code> — довжина рядка чи списку;</li>
<li><code>default</code> / <code>default_factory</code> — значення за замовчуванням;</li>
<li><code>description</code> — опис (його «бачить» LLM у JSON-схемі).</li>
</ul>

<pre><code>price: float = Field(gt=0, description="Ціна")
quantity: int = Field(default=1, ge=0)</code></pre>

</details>

### Питання 4
Що означає тип `Optional[int]` (він же `int | None`)? Чи робить `Optional` поле **необов'язковим**?

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<code>Optional[int]</code> = <code>int | None</code> означає, що значення може бути або <code>int</code>, або <code>None</code>.

⚠️ Важливо: <code>Optional</code> <b>сам по собі НЕ робить поле необов'язковим!</b> Поле стає необов'язковим лише тоді, коли є <b>значення за замовчуванням</b>.

<pre><code>a: Optional[int]              # ОБОВ'ЯЗКОВЕ (але можна передати None)
b: Optional[int] = None       # необов'язкове, за замовчуванням None</code></pre>

</details>

### Питання 5
Що таке **вкладені моделі**? З яким принципом ООП це перегукується?

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<b>Вкладена модель</b> — це коли поле моделі саме є іншою <code>BaseModel</code> (або списком таких моделей). Pydantic валідує структуру на <b>всю глибину</b> і автоматично перетворює вкладені <code>dict</code> у відповідні моделі.

Це перегукується з <b>композицією (has-a)</b> з ООП: «замовлення <b>має</b> адресу».

<pre><code>class Address(BaseModel):
    city: str

class Order(BaseModel):
    address: Address
    items: list[OrderItem]</code></pre>

</details>

### Питання 6
Навіщо потрібен `@field_validator`? Що має **повернути** метод-валідатор і як просигналізувати про помилку?

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<code>@field_validator</code> задає <b>власну перевірку/нормалізацію</b> поля, коли вбудованих обмежень <code>Field</code> не вистачає.

Метод-валідатор повинен <b>повернути значення</b> (можливо, змінене/нормалізоване) — те, що повернули, і стане значенням поля. Якщо не повернути нічого, поле стане <code>None</code>!

Щоб просигналізувати про помилку, піднімають <code>ValueError</code> (або <code>raise</code> власну) — Pydantic загорне її у <code>ValidationError</code>.

<pre><code>@field_validator("username")
@classmethod
def no_spaces(cls, v):
    if " " in v:
        raise ValueError("без пробілів")
    return v.lower()   # повертаємо нормалізоване</code></pre>

</details>

### Питання 7
Чим відрізняються `model_dump()`, `model_dump_json()`, `model_validate()` та `model_validate_json()`?

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

Це методи (де-)серіалізації:
<ul>
<li><code>model_dump()</code> — модель → <b>dict</b>;</li>
<li><code>model_dump_json()</code> — модель → <b>JSON-рядок</b>;</li>
<li><code>model_validate(data)</code> — <b>dict</b> → модель (з валідацією);</li>
<li><code>model_validate_json(s)</code> — <b>JSON-рядок</b> → модель (з валідацією).</li>
</ul>

Перші два «виводять» дані з моделі, останні два «вводять» дані в модель, перевіряючи їх.

(У Pydantic v1 це були <code>.dict()</code> / <code>.json()</code> / <code>parse_obj</code> — у v2 вони застарілі.)

</details>

### Питання 8
Як Pydantic застосовують для **структурованих виходів LLM**? Що таке `model_json_schema()` і навіщо він тут?

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

Pydantic-модель виступає <b>контрактом</b> для LLM: вона описує бажану форму відповіді (поля, типи, <code>description</code>) і <b>валідує</b> те, що реально повернула модель.

Типовий потік: просимо LLM повернути JSON → ловимо рядок через <code>model_validate_json()</code> → якщо модель «зґалюцинувала» (наприклад, оцінка поза межами), отримуємо <code>ValidationError</code> і можемо повторити запит.

<code>model_json_schema()</code> повертає <b>JSON-схему</b> моделі — її зручно передати LLM в інструкції або як опис інструмента (tool), щоб модель знала очікувану структуру.

</details>

---
## Частина 2. Знайди помилку

У кожному фрагменті коду є щонайменше одна помилка. Поясніть, у чому вона полягає, і напишіть виправлений код.

### Питання 9
Очікували `ValidationError` на поганому віці, але об'єкт спокійно створюється, а `age` лишається рядком. Чому валідація «не працює»?

In [ ]:
class User:
    name: str
    age: int


u = User()
u.name = "Іван"
u.age = "не число"   # очікували помилку — її немає
print(u.age, type(u.age))

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

<code>User</code> — <b>звичайний клас</b>, він <b>не успадковує</b> <code>BaseModel</code>. Анотації типів самі по собі нічого не перевіряють — це лише підказки. Тому валідації немає взагалі.

<pre><code>from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

u = User(name="Іван", age="не число")  # тепер буде ValidationError</code></pre>

</details>

### Питання 10
Після створення об'єкта поле `username` чомусь дорівнює `None`, хоча ми передали значення. Що не так із валідатором?

In [ ]:
class Account(BaseModel):
    username: str

    @field_validator("username")
    @classmethod
    def normalize(cls, v):
        v = v.strip().lower()
        # тут мало б щось бути...


a = Account(username="  Ivan  ")
print(repr(a.username))

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

Валідатор <b>не повертає значення</b>. Те, що поверне метод-валідатор, і стає значенням поля; якщо нічого не повернути — повертається <code>None</code>, тож <code>username</code> = <code>None</code>.

<pre><code>@field_validator("username")
@classmethod
def normalize(cls, v):
    return v.strip().lower()   # ОБОВ'ЯЗКОВО повернути значення</code></pre>

</details>

### Питання 11
Клас навіть не оголошується — Python одразу кидає `PydanticUserError` ще до створення об'єктів. Чому?

In [ ]:
class Person(BaseModel):
    name: str
    age = Field(gt=0)     # хотіли обмежити вік


p = Person(name="Іван", age=30)
print(p)

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

У поля <code>age</code> <b>немає анотації типу</b> (написано <code>age = Field(...)</code> замість <code>age: int = Field(...)</code>). Pydantic v2 <b>вимагає анотацію</b> для кожного поля й одразу при оголошенні класу піднімає <code>PydanticUserError: Field 'age' requires a type annotation</code>.

<pre><code>class Person(BaseModel):
    name: str
    age: int = Field(gt=0)   # потрібна анотація типу!</code></pre>

(Те саме станеться навіть для звичайного присвоєння без анотації, напр. <code>age = 5</code> — Pydantic v2 не дозволяє неанотованих полів.)

</details>

### Питання 12
Очікували, що поле `phone` стане необов'язковим (бо `Optional`), але без нього код падає з `ValidationError`. Чому?

In [ ]:
class Contact(BaseModel):
    name: str
    phone: Optional[str]    # думали — необов'язкове


c = Contact(name="Іван")   # ValidationError: phone обов'язкове
print(c)

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

<code>Optional[str]</code> означає лише «може бути <code>str</code> або <code>None</code>», але <b>не робить поле необов'язковим</b>. Без значення за замовчуванням поле лишається <b>обов'язковим</b> (просто йому дозволено бути <code>None</code>).

<pre><code>class Contact(BaseModel):
    name: str
    phone: Optional[str] = None   # тепер справді необов'язкове</code></pre>

</details>

### Питання 13
`u.address.city` падає з `AttributeError: 'dict' object has no attribute 'city'`, хоча дані ми передали правильні. У чому проблема?

In [ ]:
class Address(BaseModel):
    city: str


class User(BaseModel):
    name: str
    address: dict           # тип поля


u = User(name="Олена", address={"city": "Львів"})
print(u.address.city)       # AttributeError

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

Поле <code>address</code> анотоване як звичайний <code>dict</code>, тому Pydantic <b>залишив його словником</b> і не перетворив у модель. У <code>dict</code> немає атрибута <code>city</code> (лише ключ <code>"city"</code>), тож звертання через крапку падає.

Щоб отримати <b>вкладену модель</b> з валідацією та доступом через крапку, типом поля має бути сам клас моделі:

<pre><code>class User(BaseModel):
    name: str
    address: Address        # вкладена модель, а не dict

u.address.city              # тепер працює</code></pre>

</details>

### Питання 14
Клас не вдається навіть **оголосити** — Python кидає помилку ще до створення об'єктів. Що не так із валідатором?

In [ ]:
class Order(BaseModel):
    total: float

    @field_validator("totl")     # ?
    @classmethod
    def check_total(cls, v):
        if v < 0:
            raise ValueError("total не може бути від'ємним")
        return v

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

У <code>@field_validator</code> вказано <b>неіснуюче поле</b> <code>"totl"</code> (друкарська помилка замість <code>"total"</code>). Pydantic перевіряє це під час визначення класу й піднімає помилку, бо валідатор посилається на поле, якого немає.

<pre><code>@field_validator("total")   # правильна назва поля
@classmethod
def check_total(cls, v):
    if v < 0:
        raise ValueError("total не може бути від'ємним")
    return v</code></pre>

</details>

### Питання 15
Код працює, але Pydantic видає попередження `PydanticDeprecatedSince20`. Який застарілий (v1) API тут використано і як переписати на v2?

In [ ]:
from pydantic import BaseModel, validator


class User(BaseModel):
    email: str

    @validator("email")
    def must_have_at(cls, v):
        if "@" not in v:
            raise ValueError("email має містити @")
        return v

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

<code>@validator</code> — декоратор зі <b>стилю Pydantic v1</b>. Код ще працює, але v2 позначає його застарілим (<code>PydanticDeprecatedSince20</code>) і колись приберуть. Сучасний відповідник — <code>@field_validator</code>, причому валідатор прийнято позначати <code>@classmethod</code>.

<pre><code>from pydantic import BaseModel, field_validator

class User(BaseModel):
    email: str

    @field_validator("email")
    @classmethod
    def must_have_at(cls, v):
        if "@" not in v:
            raise ValueError("email має містити @")
        return v</code></pre>

</details>

---
## Частина 3. Що виведе код?

Спочатку запишіть свою відповідь у клітинці *«Ваша відповідь»*, і лише потім запустіть код і перевірте себе. Якщо помилилися — поясніть собі, чому код працює інакше.

### Питання 16

In [ ]:
class Item(BaseModel):
    name: str
    price: int
    in_stock: bool


i = Item(name="Ручка", price="15", in_stock="true")
print(i.price, type(i.price).__name__)
print(i.in_stock, type(i.in_stock).__name__)

✍️ *Ваша відповідь (що буде виведено і чому):*


<details>
<summary>💡 Відповідь</summary>

<pre><code>15 int
True bool</code></pre>

Pydantic <b>розумно приводить типи</b>: рядок <code>"15"</code> → ціле <code>15</code>, а рядок <code>"true"</code> → булеве <code>True</code>. Тому типи стають <code>int</code> і <code>bool</code>, а не <code>str</code>.

</details>

### Питання 17

In [ ]:
class Config(BaseModel):
    host: str = "localhost"
    port: int = 8000
    debug: bool = False


c = Config(port=5432)
print(c.model_dump())

✍️ *Ваша відповідь (що буде виведено і чому):*


<details>
<summary>💡 Відповідь</summary>

<pre><code>{'host': 'localhost', 'port': 5432, 'debug': False}</code></pre>

Поля з <b>значеннями за замовчуванням</b> необов'язкові. Ми передали лише <code>port=5432</code>, тож <code>host</code> і <code>debug</code> взяли значення за замовчуванням. <code>model_dump()</code> повертає всю модель як <code>dict</code>.

</details>

### Питання 18

In [ ]:
class Address(BaseModel):
    city: str


class User(BaseModel):
    name: str
    address: Address


u = User.model_validate({
    "name": "Олена",
    "address": {"city": "Львів"},
})
print(u.address.city)
print(type(u.address).__name__)

✍️ *Ваша відповідь (що буде виведено і чому):*


<details>
<summary>💡 Відповідь</summary>

<pre><code>Львів
Address</code></pre>

Поле <code>address</code> — це <b>вкладена модель</b>. Pydantic автоматично перетворив переданий <code>dict</code> на об'єкт <code>Address</code> (а не лишив словником), тому <code>u.address.city</code> працює, а тип — <code>Address</code>.

</details>

### Питання 19

In [ ]:
class Account(BaseModel):
    username: str

    @field_validator("username")
    @classmethod
    def clean(cls, v):
        return v.strip().lower()


a = Account(username="  IvanUA  ")
print(repr(a.username))

✍️ *Ваша відповідь (що буде виведено і чому):*


<details>
<summary>💡 Відповідь</summary>

<pre><code>'ivanua'</code></pre>

Валідатор не лише перевіряє, а й <b>нормалізує</b> значення: <code>strip()</code> прибирає пробіли по краях, <code>lower()</code> робить малими літерами. Повернене з валідатора значення стає значенням поля.

</details>

### Питання 20
Що виведе цей фрагмент (зокрема — скільки помилок у `ValidationError`)?

In [ ]:
class Order(BaseModel):
    product: str = Field(min_length=1)
    quantity: int = Field(gt=0)
    price: float = Field(gt=0)


try:
    Order(product="", quantity=-3, price="дорого")
except ValidationError as e:
    print("Кількість помилок:", e.error_count())

✍️ *Ваша відповідь (що буде виведено і чому):*


<details>
<summary>💡 Відповідь</summary>

<pre><code>Кількість помилок: 3</code></pre>

Усі три поля некоректні одночасно: <code>product</code> порушує <code>min_length=1</code> (порожній рядок), <code>quantity=-3</code> порушує <code>gt=0</code>, а <code>"дорого"</code> не приводиться до <code>float</code>. Pydantic <b>збирає всі помилки разом</b> (не падає на першій), тож <code>error_count()</code> поверне <code>3</code>.

Тут поєднано все: <b>обмеження Field</b>, <b>приведення типів</b> і <b>агрегація помилок</b>.

</details>